In [14]:
import requests
import pandas as pd

BASE_URL = "https://middleware.danubeproperties.com/api/marketing_collateral"
HEADERS = {
    "x_api_key": "hffghfgVb$)7yV;e0N<{e1t@,_S{_LNovryojjUpJTO#-sD4gE*HHcpBX(Q%KEK6ATy6lQK",
    "Content-Type": "application/json",
}

# ── One-time load ─────────────────────────────────────────────────────────────
resp = requests.post(BASE_URL, headers=HEADERS, timeout=60)
resp.raise_for_status()
raw = resp.json()

items = raw["data"]["marketingCollateral"]
print(f"Total projects: {len(items)}")


Total projects: 39


In [15]:
# ── Project-level DataFrame (one row per project) ────────────────────────────
rows = []
for item in items:
    rows.append({
        "marketing_collateral_id": item.get("marketing_collateral_id"),
        "project_id":              item.get("project_id"),
        "project_title":           item.get("project_title"),
        "project_location":        item.get("project_location_subtitle"),
        "thumb_full":              item.get("project_thumb_full"),
        "folders":                 [r["mc_row_folder"] for r in item.get("mc_collateral_rows", [])],
        "total_assets":            sum(len(row) for row in item.get("marketing_collateral_row_assets", [])),
    })

df_mc = pd.DataFrame(rows)
df_mc


,marketing_collateral_id,project_id,project_title,project_location,thumb_full,folders,total_assets
0,148487,148480,Greenz,"Academic City, DUBAI",https://danubeproperties.com/wp-content/upload...,"[amenities, interiors, exteriors, floor-plans,...",79
1,148529,139525,Breez,DUBAI MARITIME CITY,https://danubeproperties.com/wp-content/upload...,"[amenities, exteriors, interiors, brochures, f...",38
2,148528,144196,Shahrukhz,Sheikh Zayed Road,https://danubeproperties.com/wp-content/upload...,"[amenities, exteriors, interiors, brochures, f...",32
3,148527,146620,Serenz,Jumeirah Village Circle,https://danubeproperties.com/wp-content/upload...,"[amenities, exteriors, interiors, brochures, s...",52
4,148526,135464,Aspirz,"Sports City, Dubai",https://danubeproperties.com/wp-content/upload...,"[brochures, factsheet, floor-plans, renders, v...",45
5,148525,131949,Sparklz,"Al Furjan, DUBAI",https://danubeproperties.com/wp-content/upload...,"[amenities, exteriors, interiors, brochures, f...",42
6,151207,19775,DIAMONDZ,"Jumeirah Lake Towers, DUBAI",https://danubeproperties.com/wp-content/upload...,"[brochures, floor-plans, renders, walkthrough,...",65
7,148524,130541,Timez,"Dubai Silicon Oasis, DUBAI",https://danubeproperties.com/wp-content/upload...,"[amenities, exteriors, interiors, brochures, f...",34
8,148523,28342,Bayz 102,"BUSINESS BAY, DUBAI",https://danubeproperties.com/wp-content/upload...,"[amenities, exteriors, interiors, brochures, f...",48
9,148522,28760,Oasiz,"Dubai Silicon Oasis, DUBAI",https://danubeproperties.com/wp-content/upload...,"[amenities, exteriors, interiors, brochures, f...",40


In [17]:
# ── Flat assets DataFrame (one row per file) ─────────────────────────────────
asset_rows = []
for item in items:
    for row_group in item.get("marketing_collateral_row_assets", []):
        for asset in row_group:
            if not isinstance(asset, dict):
                continue
            asset_rows.append({
                "project_id":    item.get("project_id"),
                "project_title": item.get("project_title"),
                "folder_key":    asset.get("folder_key"),
                "file_name":     asset.get("file_name"),
                "mime_type":     asset.get("mime_type"),
                "size_bytes":    asset.get("size"),
                "url":           asset.get("url"),
                "uploaded_at":   asset.get("uploaded_at"),
            })

df_assets = pd.DataFrame(asset_rows)
df_assets["uploaded_at"] = pd.to_datetime(df_assets["uploaded_at"])
print(f"Total assets: {len(df_assets)}")
df_assets.head(10)


Total assets: 868


,project_id,project_title,folder_key,file_name,mime_type,size_bytes,url,uploaded_at
0,148480,Greenz,amenities,13.jpg,image/jpeg,3879587,https://danubeproperties.com/marketing-collate...,2026-04-01 09:43:45
1,148480,Greenz,amenities,Greenz_Amenities_11.jpg,image/jpeg,4656240,https://danubeproperties.com/marketing-collate...,2026-04-01 09:43:45
2,148480,Greenz,amenities,Greenz_Amenities_13.jpg,image/jpeg,3879587,https://danubeproperties.com/marketing-collate...,2026-04-01 09:43:45
3,148480,Greenz,amenities,Greenz_Amenities_15.jpg,image/jpeg,4574099,https://danubeproperties.com/marketing-collate...,2026-04-01 09:43:45
4,148480,Greenz,amenities,Greenz_Amenities_16.jpg,image/jpeg,4320862,https://danubeproperties.com/marketing-collate...,2026-04-01 09:43:45
5,148480,Greenz,amenities,Greenz_Amenities_17.png,image/png,37182577,https://danubeproperties.com/marketing-collate...,2026-04-01 09:43:45
6,148480,Greenz,amenities,Greenz_Amenities_18.jpg,image/jpeg,2785404,https://danubeproperties.com/marketing-collate...,2026-04-01 09:43:45
7,148480,Greenz,amenities,Greenz_Amenities_19.jpg,image/jpeg,3062758,https://danubeproperties.com/marketing-collate...,2026-04-01 09:43:45
8,148480,Greenz,amenities,Greenz_Amenities_20.jpg,image/jpeg,2762200,https://danubeproperties.com/marketing-collate...,2026-04-01 09:43:45
9,148480,Greenz,amenities,Greenz_Amenities_22.png,image/png,35967924,https://danubeproperties.com/marketing-collate...,2026-04-01 09:43:45


In [18]:
df_mc.to_csv("marketing_collateral_projects.csv", index=False)
df_assets.to_csv("marketing_collateral_assets.csv", index=False)
print("Saved.")


Saved.
